# LeetCode 832 — Flipping an Image bằng Python

**Link đề:** https://leetcode.com/problems/flipping-an-image/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, Bit Manipulation, Matrix, Simulation

```python
class Solution:
    def flipAndInvertImage(self, image: list[list[int]]) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Làm quen với **ma trận** biểu diễn bằng list lồng nhau `list[list[int]]`: truy cập `image[r][c]`, duyệt từng hàng.
2. Hai thao tác cơ bản: **lật ngang** (đảo hàng: `row[::-1]`, `reversed`) và **đảo bit** (`1 - x`, `x ^ 1`).
3. Đi từ cách "làm đúng hai bước" (tạo ma trận mới) → list comprehension → **hai con trỏ tại chỗ** (in-place) làm hai việc trong **một** lượt.
4. Hiểu bẫy **tham chiếu chung (aliasing)** khi tạo/sao chép list lồng nhau.
5. Trace, chứng minh, test biên (`n = 1`, `n` lẻ có phần tử giữa) và luyện biến thể xoay/lật ma trận.

## 1. Đề bài và ví dụ

Cho ma trận nhị phân vuông `n × n` tên `image`. Hãy **lật ngang** rồi **đảo** ảnh, trả về kết quả.

- **Lật ngang (flip horizontally):** đảo ngược từng hàng. `[1,1,0] → [0,1,1]`.
- **Đảo (invert):** thay `0` thành `1` và `1` thành `0`. `[0,1,1] → [1,0,0]`.

**Ví dụ 1:** `[[1,1,0],[1,0,1],[0,0,0]]`
- Lật: `[[0,1,1],[1,0,1],[0,0,0]]`
- Đảo: `[[1,0,0],[0,1,0],[1,1,1]]` ← đáp án

**Ví dụ 2:** `[[1,1,0,0],[1,0,0,1],[0,1,1,1],[1,0,1,0]]` → `[[1,1,0,0],[0,1,1,0],[0,0,0,1],[1,0,1,0]]`

**Constraints:** `n == len(image) == len(image[i])`; `1 <= n <= 20`; mỗi phần tử là `0` hoặc `1`.

In [1]:
image = [[1, 1, 0], [1, 0, 1], [0, 0, 0]]
for row in image:
    print(row)

[1, 1, 0]
[1, 0, 1]
[0, 0, 0]


## 2. Nền tảng Python cần dùng

### 2.1. Ma trận = list các hàng

`image[r]` là hàng thứ `r` (một list); `image[r][c]` là phần tử ở hàng `r`, cột `c`. `len(image)` là số hàng, `len(image[0])` là số cột.

In [2]:
image = [[1, 1, 0], [1, 0, 1], [0, 0, 0]]
print("hàng 0:", image[0])
print("phần tử hàng 1, cột 2:", image[1][2])
print("số hàng:", len(image), "| số cột:", len(image[0]))
for r in range(len(image)):
    for c in range(len(image[0])):
        print(image[r][c], end=" ")
    print()

hàng 0: [1, 1, 0]
phần tử hàng 1, cột 2: 1
số hàng: 3 | số cột: 3
1 1 0 
1 0 1 
0 0 0 


### 2.2. Đảo một list: `row[::-1]`, `reversed(row)`, `row.reverse()`

| Cách | Tạo list mới? | Kết quả |
|---|---|---|
| `row[::-1]` | có | list đảo |
| `list(reversed(row))` | có | list đảo (`reversed` trả về iterator) |
| `row.reverse()` | **không** — sửa tại chỗ | trả về `None` |

In [3]:
row = [1, 1, 0]
print(row[::-1], row)
print(list(reversed(row)), row)
ret = row.reverse()
print(ret, row, "<- reverse() sửa tại chỗ và trả về None")

[0, 1, 1] [1, 1, 0]
[0, 1, 1] [1, 1, 0]
None [0, 1, 1] <- reverse() sửa tại chỗ và trả về None


### 2.3. Đảo bit 0/1: `1 - x`, `x ^ 1`, `not x`

- `1 - x`: `1 - 0 = 1`, `1 - 1 = 0`.
- `x ^ 1`: toán tử **XOR** (`^`) — bit kết quả là 1 khi hai bit **khác nhau**. `0 ^ 1 = 1`, `1 ^ 1 = 0`.
- `int(not x)` cũng được nhưng kém gọn.

> Cẩn thận: `~x` trong Python **không** phải đảo 0/1! `~0 = -1`, `~1 = -2` (đảo mọi bit của số nguyên có dấu).

In [4]:
for x in [0, 1]:
    print(f"x={x}: 1-x={1 - x}, x^1={x ^ 1}, int(not x)={int(not x)}, ~x={~x}")

x=0: 1-x=1, x^1=1, int(not x)=1, ~x=-1
x=1: 1-x=0, x^1=0, int(not x)=0, ~x=-2


### 2.4. Hoán đổi hai biến: `a, b = b, a`

Python cho phép gán nhiều biến cùng lúc; vế phải được tính xong **trước** rồi mới gán. Dùng để hoán đổi `row[l]` và `row[r]` mà không cần biến tạm.

In [5]:
row = [1, 2, 3, 4]
l, r = 0, 3
row[l], row[r] = row[r], row[l]
print(row)

[4, 2, 3, 1]


### 2.5. Bẫy tham chiếu chung (aliasing)

`[[0] * 3] * 3` tạo 3 "hàng" nhưng **cùng là một list** → sửa một hàng thì cả 3 đổi. Muốn tạo ma trận độc lập, dùng list comprehension `[[0] * 3 for _ in range(3)]`. Tương tự, `new = image` **không** sao chép — chỉ là tên khác cho cùng đối tượng.

In [6]:
bad = [[0] * 3] * 3
bad[0][0] = 9
print("bad :", bad)
good = [[0] * 3 for _ in range(3)]
good[0][0] = 9
print("good:", good)

image = [[1, 0], [0, 1]]
alias = image
alias[0][0] = 7
print("image sau khi sửa alias:", image)
copy = [row[:] for row in image]   # sao chép từng hàng
copy[0][0] = 5
print("image sau khi sửa copy :", image, "| copy:", copy)

bad : [[9, 0, 0], [9, 0, 0], [9, 0, 0]]
good: [[9, 0, 0], [0, 0, 0], [0, 0, 0]]
image sau khi sửa alias: [[7, 0], [0, 1]]
image sau khi sửa copy : [[7, 0], [0, 1]] | copy: [[5, 0], [0, 1]]


## 3. Cách 1 — Làm đúng hai bước như đề (tạo ma trận mới)

Bước 1: lật từng hàng. Bước 2: đảo từng phần tử. Viết thẳng như đề mô tả — dễ đúng nhất.

In [7]:
def flip_invert_two_steps(image: list[list[int]]) -> list[list[int]]:
    flipped = []
    for row in image:
        flipped.append(row[::-1])           # bước 1: lật
    result = []
    for row in flipped:
        new_row = []
        for x in row:
            new_row.append(1 - x)           # bước 2: đảo
        result.append(new_row)
    return result

img1 = [[1, 1, 0], [1, 0, 1], [0, 0, 0]]
print(flip_invert_two_steps(img1))
print("ảnh gốc không đổi:", img1)

[[1, 0, 0], [0, 1, 0], [1, 1, 1]]
ảnh gốc không đổi: [[1, 1, 0], [1, 0, 1], [0, 0, 0]]


### Nhận xét

- Time `O(n²)` — mỗi ô được đụng tới hằng số lần. Đây đã là tối ưu về thời gian (phải xem mọi ô).
- Space `O(n²)` cho ma trận phụ `flipped` và `result`.

Câu hỏi để cải tiến: **có thể làm tại chỗ, một lượt, không cần ma trận phụ không?**

## 4. Cách 2 — List comprehension: gộp hai bước trong một biểu thức

Với mỗi hàng, `reversed(row)` cho phần tử theo thứ tự ngược; đảo ngay khi lấy ra: `[1 - x for x in reversed(row)]`. Bên ngoài là một comprehension nữa cho các hàng.

In [8]:
def flip_invert_comprehension(image: list[list[int]]) -> list[list[int]]:
    return [[1 - x for x in reversed(row)] for row in image]

print(flip_invert_comprehension([[1, 1, 0], [1, 0, 1], [0, 0, 0]]))
print(flip_invert_comprehension([[1, 1, 0, 0], [1, 0, 0, 1], [0, 1, 1, 1], [1, 0, 1, 0]]))

[[1, 0, 0], [0, 1, 0], [1, 1, 1]]
[[1, 1, 0, 0], [0, 1, 1, 0], [0, 0, 0, 1], [1, 0, 1, 0]]


## 5. Insight — Lật và đảo **giao hoán**, và làm được cùng lúc bằng hai con trỏ

**Quan sát 1:** lật rồi đảo = đảo rồi lật (đảo tác động từng ô, không phụ thuộc vị trí). Nên thứ tự không quan trọng.

**Quan sát 2:** sau khi lật, ô ở vị trí `l` nhận giá trị cũ của ô `r = n - 1 - l`, rồi bị đảo:

```text
new[l] = 1 - old[r]
new[r] = 1 - old[l]
```

Dùng hai con trỏ `l` (từ trái) và `r` (từ phải) tiến vào giữa, mỗi bước xử lý **một cặp** — hoán đổi và đảo cùng lúc, **tại chỗ**.

**Quan sát 3 (mẹo đẹp):**
- Nếu `old[l] == old[r]` (cùng 0 hoặc cùng 1): sau khi hoán đổi vẫn như cũ, rồi đảo ⇒ **cả hai đều bị đảo**.
- Nếu `old[l] != old[r]` (một 0, một 1): hoán đổi cho `[1,0] → [0,1]`, rồi đảo ⇒ `[1,0]` — **không đổi gì!**

**Phần tử giữa** (khi `n` lẻ, `l == r`): chỉ cần đảo.

In [9]:
for a, c in [(0, 0), (1, 1), (0, 1), (1, 0)]:
    pair = [a, c]
    new = [1 - pair[1], 1 - pair[0]]
    print(f"cặp cũ {pair} -> mới {new}  {'(đảo cả hai)' if a == c else '(giữ nguyên)'}")

cặp cũ [0, 0] -> mới [1, 1]  (đảo cả hai)
cặp cũ [1, 1] -> mới [0, 0]  (đảo cả hai)
cặp cũ [0, 1] -> mới [0, 1]  (giữ nguyên)
cặp cũ [1, 0] -> mới [1, 0]  (giữ nguyên)


## 6. Lời giải chính — hai con trỏ tại chỗ (từng đoạn rồi ghép)

**Đoạn 1:** duyệt từng hàng `row` của `image`.

**Đoạn 2:** `l, r = 0, len(row) - 1`; lặp `while l <= r` (dấu `<=` để xử lý cả phần tử giữa).

**Đoạn 3:** `row[l], row[r] = 1 - row[r], 1 - row[l]` — vế phải tính xong trước nên đúng cả khi `l == r` (khi đó cả hai vế là cùng một ô, kết quả là `1 - row[l]`).

**Đoạn 4:** `l += 1; r -= 1`. Trả về `image` (đã sửa tại chỗ).

In [10]:
def flip_invert_inplace(image: list[list[int]]) -> list[list[int]]:
    for row in image:
        l, r = 0, len(row) - 1
        while l <= r:
            row[l], row[r] = 1 - row[r], 1 - row[l]
            l += 1
            r -= 1
    return image

img = [[1, 1, 0], [1, 0, 1], [0, 0, 0]]
print(flip_invert_inplace(img))
print("img đã bị sửa:", img)

[[1, 0, 0], [0, 1, 0], [1, 1, 1]]
img đã bị sửa: [[1, 0, 0], [0, 1, 0], [1, 1, 1]]


## 7. TRACE từng bước

### 7.1. Bảng trace cho một hàng `[1, 0, 1, 1, 0]` (n = 5, lẻ)

| bước | l | r | row[l] cũ | row[r] cũ | gán row[l] = 1−row[r] | gán row[r] = 1−row[l] | row sau |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 4 | 1 | 0 | 1 | 0 | [1,0,1,1,0] (khác nhau → giữ) |
| 2 | 1 | 3 | 0 | 1 | 0 | 1 | [1,0,1,1,0] (khác nhau → giữ) |
| 3 | 2 | 2 | 1 | 1 | 0 | 0 | [1,0,0,1,0] (phần tử giữa → đảo) |

Kiểm tra bằng hai bước: lật `[0,1,1,0,1]`, đảo `[1,0,0,1,0]` ✓.

In [11]:
def trace_row(row: list[int]) -> list[int]:
    row = row[:]                          # sao chép để không sửa input của người gọi
    l, r = 0, len(row) - 1
    step = 1
    print("ban đầu:", row)
    while l <= r:
        a, c = row[l], row[r]
        row[l], row[r] = 1 - c, 1 - a
        note = "phần tử giữa" if l == r else ("giống nhau -> đảo cả hai" if a == c else "khác nhau -> giữ nguyên")
        print(f"bước {step}: l={l} r={r} cũ=({a},{c}) -> row={row}  [{note}]")
        l += 1; r -= 1; step += 1
    return row

trace_row([1, 0, 1, 1, 0]);
print()
trace_row([1, 1, 0, 0]);

ban đầu: [1, 0, 1, 1, 0]
bước 1: l=0 r=4 cũ=(1,0) -> row=[1, 0, 1, 1, 0]  [khác nhau -> giữ nguyên]
bước 2: l=1 r=3 cũ=(0,1) -> row=[1, 0, 1, 1, 0]  [khác nhau -> giữ nguyên]
bước 3: l=2 r=2 cũ=(1,1) -> row=[1, 0, 0, 1, 0]  [phần tử giữa]

ban đầu: [1, 1, 0, 0]
bước 1: l=0 r=3 cũ=(1,0) -> row=[1, 1, 0, 0]  [khác nhau -> giữ nguyên]
bước 2: l=1 r=2 cũ=(1,0) -> row=[1, 1, 0, 0]  [khác nhau -> giữ nguyên]


### 7.2. Trace cả ma trận ví dụ 1

In [12]:
def trace_matrix(image):
    out = []
    for i, row in enumerate(image):
        print(f"--- hàng {i} ---")
        out.append(trace_row(row))
    print("Kết quả:", out)
    return out

trace_matrix([[1, 1, 0], [1, 0, 1], [0, 0, 0]]);

--- hàng 0 ---
ban đầu: [1, 1, 0]
bước 1: l=0 r=2 cũ=(1,0) -> row=[1, 1, 0]  [khác nhau -> giữ nguyên]
bước 2: l=1 r=1 cũ=(1,1) -> row=[1, 0, 0]  [phần tử giữa]
--- hàng 1 ---
ban đầu: [1, 0, 1]
bước 1: l=0 r=2 cũ=(1,1) -> row=[0, 0, 0]  [giống nhau -> đảo cả hai]
bước 2: l=1 r=1 cũ=(0,0) -> row=[0, 1, 0]  [phần tử giữa]
--- hàng 2 ---
ban đầu: [0, 0, 0]
bước 1: l=0 r=2 cũ=(0,0) -> row=[1, 0, 1]  [giống nhau -> đảo cả hai]
bước 2: l=1 r=1 cũ=(0,0) -> row=[1, 1, 1]  [phần tử giữa]
Kết quả: [[1, 0, 0], [0, 1, 0], [1, 1, 1]]


## 8. Code nộp LeetCode

In [13]:
class Solution:
    def flipAndInvertImage(self, image: list[list[int]]) -> list[list[int]]:
        for row in image:
            l, r = 0, len(row) - 1
            while l <= r:
                row[l], row[r] = row[r] ^ 1, row[l] ^ 1
                l += 1
                r -= 1
        return image

sol = Solution()
assert sol.flipAndInvertImage([[1, 1, 0], [1, 0, 1], [0, 0, 0]]) == [[1, 0, 0], [0, 1, 0], [1, 1, 1]]
assert sol.flipAndInvertImage([[1, 1, 0, 0], [1, 0, 0, 1], [0, 1, 1, 1], [1, 0, 1, 0]]) == \
    [[1, 1, 0, 0], [0, 1, 1, 0], [0, 0, 0, 1], [1, 0, 1, 0]]
print("Solution PASS 2 ví dụ đề")

Solution PASS 2 ví dụ đề


Bản nộp dùng `^ 1` (XOR) thay cho `1 - x` — tương đương với 0/1, đúng chủ đề *Bit Manipulation* của bài. Bản one-liner cũng được chấp nhận:

In [14]:
class SolutionOneLiner:
    def flipAndInvertImage(self, image: list[list[int]]) -> list[list[int]]:
        return [[x ^ 1 for x in row[::-1]] for row in image]

print(SolutionOneLiner().flipAndInvertImage([[1, 1, 0], [1, 0, 1], [0, 0, 0]]))

[[1, 0, 0], [0, 1, 0], [1, 1, 1]]


## 9. Vì sao đúng? Độ phức tạp

**Đúng:** Với một hàng độ dài `n`, kết quả mong muốn là `new[j] = 1 - old[n - 1 - j]` với mọi `j`.

**Invariant của vòng `while`:** trước mỗi lần lặp, các ô ở vị trí `< l` và `> r` đã có giá trị đúng `new[...]`, còn các ô trong `[l, r]` vẫn là giá trị cũ. Mỗi lần lặp gán đúng cho cặp `(l, r)` (vì `r = n - 1 - l`) bằng giá trị cũ của nhau — vế phải tính **trước** khi gán, nên không đọc phải giá trị đã sửa. Khi `l == r`, phép gán cho ra `1 - old[l]` = đúng `new[l]` (vì `n - 1 - l = l`). Vòng dừng khi `l > r` — mọi ô đã đúng.

**Độ phức tạp:** ma trận `n × n`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Hai bước, ma trận mới | `O(n²)` | `O(n²)` |
| 2. List comprehension | `O(n²)` | `O(n²)` (ma trận kết quả mới) |
| 3. Hai con trỏ tại chỗ | `O(n²)` — mỗi hàng `⌈n/2⌉` vòng | `O(1)` |
| 4. Tại chỗ + mẹo "chỉ đảo khi bằng nhau" | `O(n²)` | `O(1)` |

## 10. Cách 4 — Mẹo "chỉ đảo khi hai đầu bằng nhau"

Từ quan sát 3: nếu `row[l] == row[r]` thì đảo cả hai; nếu khác nhau thì **không làm gì**. Phần tử giữa luôn đảo (trường hợp `l == r` tự rơi vào nhánh "bằng nhau" và đảo một lần — nhưng cẩn thận, phải đảo **một** lần chứ không phải hai!).

In [15]:
def flip_invert_equal_trick(image: list[list[int]]) -> list[list[int]]:
    for row in image:
        l, r = 0, len(row) - 1
        while l < r:
            if row[l] == row[r]:
                row[l] ^= 1
                row[r] ^= 1
            l += 1
            r -= 1
        if l == r:                 # n lẻ: phần tử giữa
            row[l] ^= 1
    return image

print(flip_invert_equal_trick([[1, 1, 0], [1, 0, 1], [0, 0, 0]]))

[[1, 0, 0], [0, 1, 0], [1, 1, 1]]


`x ^= 1` là viết tắt của `x = x ^ 1` (tương tự `+=`).

Nếu viết `while l <= r` và đảo cả `row[l]`, `row[r]` khi bằng nhau, thì tại `l == r` ô giữa bị đảo **hai lần** → trở về như cũ → sai. Đó là lý do tách riêng xử lý phần tử giữa.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Hai ví dụ đề.
2. `n = 1`: `[[0]]`, `[[1]]`.
3. `n` chẵn và lẻ.
4. Ma trận toàn 0, toàn 1.
5. **Tính chất:** áp dụng hai lần sẽ ra lại ảnh gốc (phép biến đổi là "tự nghịch đảo").

In [16]:
import copy

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

def run(img):                       # chạy trên bản sao để không sửa dữ liệu test
    return Solution().flipAndInvertImage(copy.deepcopy(img))

check("đề 1", run([[1, 1, 0], [1, 0, 1], [0, 0, 0]]), [[1, 0, 0], [0, 1, 0], [1, 1, 1]])
check("đề 2", run([[1, 1, 0, 0], [1, 0, 0, 1], [0, 1, 1, 1], [1, 0, 1, 0]]),
      [[1, 1, 0, 0], [0, 1, 1, 0], [0, 0, 0, 1], [1, 0, 1, 0]])
check("n=1 (0)", run([[0]]), [[1]])
check("n=1 (1)", run([[1]]), [[0]])
check("toàn 0", run([[0, 0], [0, 0]]), [[1, 1], [1, 1]])
check("toàn 1, n=3", run([[1, 1, 1], [1, 1, 1], [1, 1, 1]]), [[0, 0, 0], [0, 0, 0], [0, 0, 0]])

PASS đề 1: [[1, 0, 0], [0, 1, 0], [1, 1, 1]]
PASS đề 2: [[1, 1, 0, 0], [0, 1, 1, 0], [0, 0, 0, 1], [1, 0, 1, 0]]
PASS n=1 (0): [[1]]
PASS n=1 (1): [[0]]
PASS toàn 0: [[1, 1], [1, 1]]
PASS toàn 1, n=3: [[0, 0, 0], [0, 0, 0], [0, 0, 0]]


`copy.deepcopy(x)` sao chép **sâu** — cả list ngoài lẫn từng list bên trong — để lời giải tại chỗ không làm hỏng dữ liệu test (mục 2.5).

### 11.1. Đối chiếu 6 cài đặt + tính chất tự nghịch đảo (có seed)

In [17]:
import random
random.seed(832)
impls = [flip_invert_two_steps, flip_invert_comprehension, flip_invert_inplace,
         flip_invert_equal_trick, Solution().flipAndInvertImage, SolutionOneLiner().flipAndInvertImage]
for _ in range(300):
    n = random.randint(1, 7)
    img = [[random.randint(0, 1) for _ in range(n)] for _ in range(n)]
    expected = flip_invert_two_steps(img)
    for f in impls:
        assert f(copy.deepcopy(img)) == expected, (img, f)
    assert flip_invert_two_steps(expected) == img       # làm hai lần -> về ảnh gốc
print("300 ma trận ngẫu nhiên: 6 cài đặt khớp nhau; áp dụng 2 lần ra ảnh gốc")

300 ma trận ngẫu nhiên: 6 cài đặt khớp nhau; áp dụng 2 lần ra ảnh gốc


## 12. Lỗi người mới hay gặp

### Lỗi 1: hoán đổi bằng hai lệnh gán riêng

```python
row[l] = 1 - row[r]
row[r] = 1 - row[l]     # row[l] đã bị ghi đè!
```

In [18]:
def wrong_swap(row):
    row = row[:]
    l, r = 0, len(row) - 1
    while l <= r:
        row[l] = 1 - row[r]
        row[r] = 1 - row[l]
        l += 1; r -= 1
    return row

print("sai :", wrong_swap([1, 1, 0]))
print("đúng:", flip_invert_comprehension([[1, 1, 0]])[0])

sai : [1, 1, 0]
đúng: [1, 0, 0]


### Lỗi 2: dùng `~x` để đảo bit

`~1 == -2`, `~0 == -1` — không phải 0/1.

In [19]:
row = [1, 0, 1]
print("sai :", [~x for x in row[::-1]])
print("đúng:", [x ^ 1 for x in row[::-1]])

sai : [-2, -1, -2]
đúng: [0, 1, 0]


### Lỗi 3: `while l < r` trong cách hai con trỏ (cách 3) → quên phần tử giữa

In [20]:
def wrong_middle(row):
    row = row[:]
    l, r = 0, len(row) - 1
    while l < r:                      # SAI: bỏ sót l == r
        row[l], row[r] = 1 - row[r], 1 - row[l]
        l += 1; r -= 1
    return row

print("sai :", wrong_middle([1, 0, 1]))
print("đúng:", [1 - x for x in [1, 0, 1][::-1]])

sai : [0, 0, 0]
đúng: [0, 1, 0]


### Lỗi 4: `row.reverse()` rồi dùng giá trị trả về

`reverse()` trả về `None` → `[1 - x for x in row.reverse()]` gây `TypeError`.

In [21]:
row = [1, 1, 0]
try:
    print([1 - x for x in row.reverse()])
except TypeError as e:
    print("TypeError:", e)

TypeError: 'NoneType' object is not iterable


## 13. Mẹo nhận dạng pattern

- "Đảo ngược mảng / hàng", "đối xứng", "so sánh hai đầu" → **hai con trỏ `l`, `r` đi vào giữa**; nhớ trường hợp `n` lẻ có phần tử giữa.
- Dữ liệu chỉ có 0/1 → nghĩ tới `x ^ 1`, `1 - x`, và các mẹo bit.
- Thao tác ma trận (lật, xoay, chuyển vị) → tách thành thao tác **trên hàng/cột** quen thuộc; kiểm tra tính chất như "làm hai lần ra lại gốc".
- Bài liên quan: *Reverse String (344)*, *Rotate Image (48)* (= chuyển vị + lật ngang), *Transpose Matrix (867)*, *Valid Palindrome (125)*.

## 14. Biến thể

### 14.1. Lật **dọc** (đảo thứ tự các hàng)

In [22]:
def flip_vertical(image):
    return [row[:] for row in image[::-1]]

print(flip_vertical([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[[7, 8, 9], [4, 5, 6], [1, 2, 3]]


### 14.2. Xoay ma trận 90° theo chiều kim đồng hồ = chuyển vị rồi lật ngang

`zip(*image)` "ghép" các hàng theo cột → cho ra các **cột** (chính là chuyển vị). Ta sẽ học kỹ ở bài 867.

In [23]:
def rotate_90(image):
    transposed = [list(col) for col in zip(*image)]
    return [row[::-1] for row in transposed]

print(rotate_90([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


### 14.3. Ảnh dạng chuỗi bit: mỗi hàng là một số nguyên

Nếu mỗi hàng `n` bit được lưu thành một số, "đảo" là `x ^ ((1 << n) - 1)` — XOR với mặt nạ toàn 1.

In [24]:
n = 3
mask = (1 << n) - 1          # 0b111 = 7
x = 0b011
print(bin(mask), bin(x), "->", bin(x ^ mask))

0b111 0b11 -> 0b100


## 15. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Dự đoán output
Không chạy, đoán: `[1, 2, 3][::-1]`, `0 ^ 1`, `1 ^ 1`, `~0`, `[[0] * 2] * 2` sau khi gán `a[0][0] = 1`.

### Bài 2 — Tính tay
Tính `flipAndInvertImage([[0, 1, 1, 1], [1, 0, 0, 0]])` (ma trận không vuông vẫn áp dụng được theo hàng).

### Bài 3 — Điền TODO cho cách hai con trỏ

In [25]:
def ex3_flip_invert(image: list[list[int]]) -> list[list[int]]:
    for row in image:
        l, r = 0, len(row) - 1
        # TODO: while ...: gán đồng thời, đảo, dịch con trỏ
        pass
    return image

# assert ex3_flip_invert([[1, 1, 0], [1, 0, 1], [0, 0, 0]]) == [[1, 0, 0], [0, 1, 0], [1, 1, 1]]

### Bài 4 — Reverse String tại chỗ (LeetCode 344)
Đảo list ký tự `s` **tại chỗ**, không dùng `[::-1]` hay `reverse()`.

In [26]:
def ex4_reverse_inplace(s: list[str]) -> None:
    # TODO: hai con trỏ
    pass

# s = list("hello"); ex4_reverse_inplace(s); assert s == list("olleh")

### Bài 5 — Đếm số ô thay đổi
Cho `image`, đếm số ô có giá trị **khác** giữa ảnh gốc và ảnh sau khi flip-invert. (Gợi ý: dùng mẹo cặp bằng/khác nhau.)

In [27]:
def ex5_count_changed(image: list[list[int]]) -> int:
    # TODO
    pass

# assert ex5_count_changed([[1, 1, 0], [1, 0, 1], [0, 0, 0]]) == 7

### Bài 6 — Mini challenge: xoay 90° **tại chỗ** (LeetCode 48)
Xoay ma trận vuông 90° chiều kim đồng hồ, sửa trực tiếp `matrix`. Gợi ý: chuyển vị tại chỗ (hoán đổi `m[i][j]` với `m[j][i]` khi `i < j`), rồi đảo từng hàng.

In [28]:
def ex6_rotate_inplace(matrix: list[list[int]]) -> None:
    # TODO
    pass

# m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; ex6_rotate_inplace(m); assert m == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 2: lật từng hàng rồi đảo.
- Bài 5: một ô thay đổi khi `new[j] != old[j]`; dùng quan sát "cặp bằng nhau → cả hai đổi; khác nhau → giữ".
- Bài 6: xoay 90° = chuyển vị + lật ngang.

**Mức 2 — cú pháp**
- Bài 3: `while l <= r: row[l], row[r] = row[r] ^ 1, row[l] ^ 1; l += 1; r -= 1`.
- Bài 4: `s[l], s[r] = s[r], s[l]` với `while l < r`.

**Mức 3 — gần code**
- Bài 5: so sánh trực tiếp: `sum(1 for r in range(n) for c in range(n) if image[r][c] != 1 - image[r][n-1-c])`.
- Bài 6: `for i in range(n): for j in range(i + 1, n): m[i][j], m[j][i] = m[j][i], m[i][j]`; sau đó `for row in m: row.reverse()`.

## 17. Đáp án tham khảo

**Bài 1:** `[3, 2, 1]`, `1`, `0`, `-1`, `[[1, 0], [1, 0]]` (hai hàng là cùng một list!).

**Bài 2:** hàng 1 lật `[1,1,1,0]` → đảo `[0,0,0,1]`; hàng 2 lật `[0,0,0,1]` → đảo `[1,1,1,0]`. Kết quả `[[0,0,0,1],[1,1,1,0]]`.

In [29]:
a = [[0] * 2] * 2
a[0][0] = 1
print([1, 2, 3][::-1], 0 ^ 1, 1 ^ 1, ~0, a)
assert Solution().flipAndInvertImage([[0, 1, 1, 1], [1, 0, 0, 0]]) == [[0, 0, 0, 1], [1, 1, 1, 0]]

def ans3_flip_invert(image):
    for row in image:
        l, r = 0, len(row) - 1
        while l <= r:
            row[l], row[r] = row[r] ^ 1, row[l] ^ 1
            l += 1; r -= 1
    return image

def ans4_reverse_inplace(s):
    l, r = 0, len(s) - 1
    while l < r:
        s[l], s[r] = s[r], s[l]
        l += 1; r -= 1

def ans5_count_changed(image):
    n = len(image[0])
    return sum(1 for row in image for c in range(n) if row[c] != 1 - row[n - 1 - c])

def ans6_rotate_inplace(m):
    n = len(m)
    for i in range(n):
        for j in range(i + 1, n):
            m[i][j], m[j][i] = m[j][i], m[i][j]
    for row in m:
        row.reverse()

assert ans3_flip_invert([[1, 1, 0], [1, 0, 1], [0, 0, 0]]) == [[1, 0, 0], [0, 1, 0], [1, 1, 1]]
s = list("hello"); ans4_reverse_inplace(s); assert s == list("olleh")
assert ans5_count_changed([[1, 1, 0], [1, 0, 1], [0, 0, 0]]) == 7
m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; ans6_rotate_inplace(m); assert m == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
assert m == rotate_90([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Đáp án bài 1-6 PASS")

[3, 2, 1] 1 0 -1 [[1, 0], [1, 0]]
Đáp án bài 1-6 PASS


## 18. Tổng kết — template nhớ nhanh và checklist submit

```python
for row in image:
    l, r = 0, len(row) - 1
    while l <= r:                                   # <= để xử lý phần tử giữa
        row[l], row[r] = row[r] ^ 1, row[l] ^ 1     # hoán đổi + đảo, gán đồng thời
        l += 1; r -= 1
return image
```

Hoặc gọn: `return [[x ^ 1 for x in row[::-1]] for row in image]`.

**Checklist trước khi submit:**
- [ ] Hoán đổi bằng **gán đồng thời** `a, b = ..., ...` (không ghi đè trước khi đọc)?
- [ ] `while l <= r` (cách 3) hoặc xử lý riêng phần tử giữa (cách 4)?
- [ ] Đảo bằng `x ^ 1` hoặc `1 - x` — **không** dùng `~x`?
- [ ] Không dùng giá trị trả về của `row.reverse()`?
- [ ] Đã test `n = 1`, `n` lẻ, `n` chẵn?